### **Foundations of Features**

Welcome to Feature Engineering. Before we start transforming and creating features, we need to understand what a feature actually is — and why the features you choose matter far more than the algorithm you pick.

**In this notebook we will cover:**
1. What features and targets are
2. Types of machine learning problems
3. Why feature engineering matters more than algorithm choice
4. The different types of features you'll encounter in real data
5. Where feature engineering fits in the full ML pipeline

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)
np.random.seed(42)

### 1. What is a Feature? What is a Target?

Think about how a real estate agent estimates a house price. They walk through the property and mentally note things down — how many bedrooms it has, how large the garden is, which neighbourhood it's in, how old the building is. All of those observations they feed into their brain to produce one number: the estimated price.

In machine learning, those observations are called **features** — they are the information we give to the model. The number the model produces (house price, in this case) is called the **target**.

You'll also hear features called input variables, predictors, or independent variables. The target goes by label, response variable, or dependent variable. They all mean the same thing.

> **A simple rule to remember:** Features = what you know. Target = what you want to predict.

In a pandas DataFrame, features are all the columns except the one you're predicting (usually stored in a variable called `X`), and the target is that one column (usually called `y`).

In [3]:
# A simple dataset: predicting house prices
data = {
    'bedrooms':   [2,      3,      4,      3,      5     ],
    'area_sqft':  [1200,   1500,   2000,   1800,   2500  ],
    'location':   ['downtown', 'suburbs', 'suburbs', 'downtown', 'rural'],
    'age_years':  [5,      10,     2,      15,     8     ],
    'price_usd':  [350000, 280000, 420000, 310000, 195000]
}
df_houses = pd.DataFrame(data)

print('Full DataFrame (features + target):')
df_houses

Full DataFrame (features + target):


,bedrooms,area_sqft,location,age_years,price_usd
0,2,1200,downtown,5,350000
1,3,1500,suburbs,10,280000
2,4,2000,suburbs,2,420000
3,3,1800,downtown,15,310000
4,5,2500,rural,8,195000


In [4]:
# Separate features (X) and target (y)
X = df_houses.drop('price_usd', axis=1)
y = df_houses['price_usd']

print('\nFeatures (X):')
X


Features (X):


,bedrooms,area_sqft,location,age_years
0,2,1200,downtown,5
1,3,1500,suburbs,10
2,4,2000,suburbs,2
3,3,1800,downtown,15
4,5,2500,rural,8


Notice how separating `X` (the features) from `y` (the target) is the very first thing we do. The model will only ever see `X` during training — it uses those columns to learn patterns and then tries to predict `y`. Keeping them separate from the start helps avoid accidentally leaking the answer into the input.

In [9]:
print('\nTarget (y):')
print(y)


Target (y):
0    350000
1    280000
2    420000
3    310000
4    195000
Name: price_usd, dtype: int64


### 2. Types of Machine Learning Problems

The type of problem you're solving shapes everything such as which features make sense, how you measure success, and what your target column looks like.

If you're trying to predict **a number** (house price, temperature, tomorrow's sales), that's a **regression** problem. The target is continuous which can be any numerical value.

If you're trying to predict **which category** something belongs to (spam or not, will this customer churn, fraud transaction or not), that's a **classification** problem. The target is a label from a fixed set.

And if there's **no target at all** then we just group similar things together which is called as **clustering** (unsupervised learning). You're not predicting anything; you're discovering structure.

| Problem Type | What you predict | Target type | Example |
|---|---|---|---|
| **Regression** | A number | Continuous | House price, temperature, stock price |
| **Classification** | A category | Discrete label | Spam or not, disease or healthy, churn or stay |
| **Clustering** | Groups | None (unsupervised) | Customer segments, document topics |

The type of problem matters because it determines how you evaluate the model.

In [12]:
# Regression example: predict house price (continuous number)
df_regression = pd.DataFrame({
    'size_sqft': [1000, 1500, 2000, 2500, 3000],
    'rooms':     [3, 4, 4, 5, 6],
    'price':     [200000, 300000, 380000, 450000, 520000]  # continuous
})
print('Regression dataset (continuous target):')
df_regression

Regression dataset (continuous target):


,size_sqft,rooms,price
0,1000,3,200000
1,1500,4,300000
2,2000,4,380000
3,2500,5,450000
4,3000,6,520000


In [14]:
# Classification example: predict spam or not (category)
df_classification = pd.DataFrame({
    'word_count':     [50,  200, 15,  300, 80 ],
    'has_link':       [1,   0,   1,   0,   1  ],
    'sender_known':   [0,   1,   0,   1,   0  ],
    'is_spam':        [1,   0,   1,   0,   1  ]  # categorical (0 or 1)
})
print('\nClassification dataset (binary target):')
df_classification


Classification dataset (binary target):


,word_count,has_link,sender_known,is_spam
0,50,1,0,1
1,200,0,1,0
2,15,1,0,1
3,300,0,1,0
4,80,1,0,1


### 3. Why Feature Engineering Matters More Than You Think

There is a famous saying in machine learning:

> *"Data and features decide the ceiling of a model's performance. Algorithms only help you approach that ceiling."*

This is one of the most important ideas in all of applied ML. It means that even the most powerful algorithm in the world cannot make up for poor features. On the other hand, a simple logistic regression trained on great features will often beat a deep neural network trained on raw, unprocessed data.

This is not just theory — Kaggle competition winners consistently report that their edge came from better feature engineering, not a better model. They often use the exact same algorithms as everyone else.

Let's see this in action. We'll create a dataset where the true relationship is: a house is "premium" if its area per room exceeds 400 square feet. A model given the raw features (area and rooms separately) has to figure out the ratio on its own. A model given the engineered feature (area ÷ rooms) has the answer handed to it.

In [17]:
np.random.seed(0)
n = 500

area  = np.random.randint(500, 3000, n).astype(float)
rooms = np.random.randint(2, 8, n).astype(float)

# True relationship: premium if area/rooms > 400
area_per_room = area / rooms  # We engineered this feature for better accuracy
y_label = (area_per_room > 400).astype(int)

In [19]:
# Dataset with raw features only
X_raw = np.column_stack([area, rooms])

X_raw_train, X_raw_test, y_train, y_test = train_test_split(X_raw, y_label, test_size=0.2, random_state=42)

# Train the same logistic regression on both
model_raw = LogisticRegression(max_iter=1000)
model_raw.fit(X_raw_train, y_train)

y_pred_raw = model_raw.predict(X_raw_test)
acc_raw = accuracy_score(y_test, y_pred_raw)

print(f'Accuracy WITHOUT feature engineering: {acc_raw:.3f}')

Accuracy WITHOUT feature engineering: 0.980


In [21]:
# Dataset with engineered feature added
X_engineered = np.column_stack([area, rooms, area_per_room])

X_eng_train, X_eng_test, y_train, y_test = train_test_split(X_engineered, y_label, test_size=0.2, random_state=42)

model_eng = LogisticRegression(max_iter=1000)
model_eng.fit(X_eng_train, y_train)

y_pred_eng = model_eng.predict(X_eng_test)
acc_eng = accuracy_score(y_test, y_pred_eng)

print(f'Accuracy WITH feature engineering:  {acc_eng:.3f}')

Accuracy WITH feature engineering:  1.000


In [23]:
print(f'\nImprovement: +{(acc_eng - acc_raw)*100:.1f}% — same algorithm, better features!')


Improvement: +2.0% — same algorithm, better features!


The result might surprise you. When we add the `area_per_room` feature — which is just `area / rooms` — the model's job becomes trivial: it just needs to find a threshold. Without it, a linear model has to discover that relationship by itself, and it often cannot. Same algorithm. Same data. Better feature → better results.

This is the whole point of feature engineering.

### 4. Types of Features in Real Data

Real-world datasets contain many different kinds of features, and understanding what type each feature is tells you how to handle it. The type determines which transformations make sense, which encoding strategy to use, and which models can work with it directly.

**Numerical features** come in two flavours. A *continuous* feature can take any value within a range — salary, temperature, house price. A *discrete* feature can only be whole numbers — number of children, page views, items in a basket.

**Categorical features** are labels or names. *Nominal* categories have no natural order: city names, colours, product types. There's no sense in saying "Paris" is greater than "Tokyo". *Ordinal* categories do have a meaningful order: T-shirt sizes (S < M < L < XL), education levels (HS < Bachelor < Master < PhD). The order matters and we need to preserve it.

**Binary features** are special cases with just two values — `is_fraud` (0 or 1), `has_insurance` (True or False).

**Text, date-time, and time series features** each have their own engineering approaches, which we'll cover in dedicated notebooks later in this series.

Knowing the type of each feature before you touch it is step one of any feature engineering workflow.

In [27]:
# A rich DataFrame with one column of each feature type
df_types = pd.DataFrame({
    # Numerical - continuous
    'salary':           [75000.0, 52000.0, 120000.0, 43000.0, 95000.0],
    # Numerical - discrete
    'num_children':     [0, 2, 1, 3, 0],
    # Categorical - nominal (no inherent order)
    'city':             ['New York', 'Berlin', 'Tokyo', 'Paris', 'Sydney'],
    # Categorical - ordinal (has a meaningful order)
    'education':        ['Bachelor', 'HS', 'PhD', 'Master', 'Bachelor'],
    # Binary indicator
    'has_insurance':    [True, False, True, False, True],
    # Text
    'review':           ['Great service!', 'Terrible experience', 'Average product', 'Love it!', 'Would not buy again'],
    # Date-time
    'signup_date':      ['2023-01-15', '2022-08-20', '2024-03-01', '2021-11-05', '2023-07-22'],
})

print('DataFrame with all feature types:')
df_types

DataFrame with all feature types:


,salary,num_children,city,education,has_insurance,review,signup_date
0,75000.0,0,New York,Bachelor,True,Great service!,2023-01-15
1,52000.0,2,Berlin,HS,False,Terrible experience,2022-08-20
2,120000.0,1,Tokyo,PhD,True,Average product,2024-03-01
3,43000.0,3,Paris,Master,False,Love it!,2021-11-05
4,95000.0,0,Sydney,Bachelor,True,Would not buy again,2023-07-22


In [29]:
print('Pandas dtypes:')
df_types.dtypes

Pandas dtypes:


salary           float64
num_children       int64
city              object
education         object
has_insurance       bool
review            object
signup_date       object
dtype: object

In [31]:
# IMPORTANT: pandas doesn't know about ordinal order!
# We have to tell it explicitly.

education_order = ['HS', 'Bachelor', 'Master', 'PhD']

df_types['education_cat'] = pd.Categorical(
    df_types['education'],
    categories=education_order,
    ordered=True  # <-- this tells pandas the order matters
)

print('Education as ordered Categorical:')
print(df_types['education_cat'])
print('\nCan now compare: PhD > Master?', df_types['education_cat'].iloc[2] > df_types['education_cat'].iloc[3])

Education as ordered Categorical:
0    Bachelor
1          HS
2         PhD
3      Master
4    Bachelor
Name: education_cat, dtype: category
Categories (4, object): ['HS' < 'Bachelor' < 'Master' < 'PhD']

Can now compare: PhD > Master? True


In [33]:
# Also: parse the date column properly
df_types['signup_date'] = pd.to_datetime(df_types['signup_date'])

print('\nAfter fixing dtypes:')
print(df_types[['salary', 'city', 'education_cat', 'signup_date']].dtypes)


After fixing dtypes:
salary                  float64
city                     object
education_cat          category
signup_date      datetime64[ns]
dtype: object


Notice the `ordered=True` flag on the Categorical. Without it, pandas treats education as just a collection of labels — it doesn't know that PhD is "above" Master. With it, we can compare them directly and any algorithm that respects ordinality will treat the levels correctly. Always be explicit about order when you know it exists.

### 5. Where Feature Engineering Fits in the ML Pipeline

Feature engineering sits right in the middle of the machine learning workflow — after you've understood and cleaned your data, and before you train your model. Here's the full picture:

```
Raw Data → EDA → Data Cleaning → Feature Engineering ← YOU ARE HERE
     → Train/Validation Split → Model Training → Evaluation → Deployment
```

One rule is critical and beginners break it constantly: **you must split your data into training and test sets BEFORE fitting any feature transformer**. If you scale or impute using statistics from the full dataset (including test rows), your test set is no longer a fair evaluation — the model has already "seen" test data through the transformer. This is called **data leakage**, and it makes your model look better than it really is.

The cleanest way to enforce this rule is with sklearn Pipelines, which chain transformers and the model together so that `fit()` is always called on training data only. Here's a quick preview — we'll go deep on Pipelines in Notebook 10.

In [37]:
from sklearn.impute import SimpleImputer

# Generate a simple numeric dataset
X_demo, y_demo = make_classification(n_samples=300, n_features=5, random_state=42)

# Introduce some missing values
X_demo_df = pd.DataFrame(X_demo, columns=[f'feat_{i}' for i in range(5)])
X_demo_df.iloc[np.random.choice(300, 30), 0] = np.nan

# Split FIRST — then build the pipeline
X_tr, X_te, y_tr, y_te = train_test_split(X_demo_df, y_demo, test_size=0.2, random_state=42)

# Pipeline: impute → scale → model
# Each step is fit only on training data when you call .fit(X_tr, y_tr)
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),  # fill missing with training mean
    ('scaler',  StandardScaler()),                # scale using training stats
    ('model',   LogisticRegression())
])

pipe.fit(X_tr, y_tr)
print(f'Pipeline accuracy on test set: {pipe.score(X_te, y_te):.3f}')
print('\nThe pipeline ensures test data never influences training transforms!')
print('We will explore Pipelines in depth in Notebook 10.')

Pipeline accuracy on test set: 1.000

The pipeline ensures test data never influences training transforms!
We will explore Pipelines in depth in Notebook 10.


See how the pipeline's `fit()` call only touches the training data `X_tr`? When we call `pipe.score(X_te, y_te)`, the pipeline applies the transformers using statistics it learned from training — it never fits again on the test data. This is the pattern we'll use throughout the entire course.

### <font color='green'>Key things to remember from this notebook</font>

Features are the inputs your model uses to learn; the target is what it's trying to predict. The type of problem (regression, classification, or clustering) shapes everything that follows. And the single most important lesson: features set the ceiling of your model's performance — no algorithm can compensate for bad features, but great features make even simple algorithms powerful.

In the next notebook we'll look at what raw data actually looks like — the missing values, outliers, and inconsistencies that need to be fixed before we can start engineering useful features.